# Exploração do modelo estático de pneu — V4

Notebook de desenvolvimento para comparar perfis PAC2002, forças e momentos em regime permanente. O notebook chama o módulo Python; não reimplementa equações. Requer NumPy e Matplotlib no kernel Jupyter (Matplotlib não faz parte das dependências de execução atuais do projeto). Os quatro arquivos TTC locais ficam em `data/local_tires/round_6_hoosier_r25b/` (ignorados pelo Git). Se os dados não estiverem nesse caminho, ajuste `TIRE_DIR`.

**Limites:** os perfis são ajustes separados em 8, 10, 12 e 14 psi; este estudo não interpola pressão. O dataset é do R25B, não do R60A da equipe. Ainda não existe validação experimental independente do módulo. Ângulos em radianos e carga positiva em N.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'python').is_dir():
    raise RuntimeError('Abra o Jupyter com a raiz do repositório como diretório de trabalho.')
sys.path.insert(0, str(ROOT / 'python'))
import tire_model_static_v4 as tire

TIRE_DIR = ROOT / 'data' / 'local_tires' / 'round_6_hoosier_r25b'
family = tire.load_profile_family(TIRE_DIR, source_axis_system='ISO_TYDEX')
print('Pressões nominais disponíveis (psi):', family.pressures_psi)
for pressure in family.pressures_psi:
    p = family.at_pressure(pressure).parameters
    print(f'{pressure:g} psi | Fz: {p["FZMIN"]:.0f}–{p["FZMAX"]:.0f} N | α: {np.rad2deg(p["ALPMIN"]):.1f}–{np.rad2deg(p["ALPMAX"]):.1f}° | γ: {np.rad2deg(p["CAMMIN"]):.1f}–{np.rad2deg(p["CAMMAX"]):.1f}° | κ: {p["KPUMIN"]:.2f}–{p["KPUMAX"]:.2f}')


## Comparação de forças e momentos em função de slip angle

A carga é definida como uma fração de `F′z0` de cada arquivo. Ajuste carga, cambagem, escorregamento longitudinal e velocidade para a manobra de interesse. O domínio deve permanecer dentro da faixa de medição declarada no `.tir`. Os sinais estão no referencial do pneu indicado pelo perfil.

In [ ]:
load_fraction = 1.0
camber_deg = 0.0
kappa = 0.0
speed_mps = min(family.at_pressure(p).parameters['LONGVL'] for p in family.pressures_psi)
alpha_min_deg = max(np.rad2deg(family.at_pressure(p).parameters['ALPMIN']) for p in family.pressures_psi)
alpha_max_deg = min(np.rad2deg(family.at_pressure(p).parameters['ALPMAX']) for p in family.pressures_psi)
alpha_deg = np.linspace(alpha_min_deg, alpha_max_deg, 121)

curves = {}
for pressure in family.pressures_psi:
    profile = family.at_pressure(pressure)
    fz = load_fraction * profile.parameters['LFZO'] * profile.parameters['FNOMIN']
    curves[pressure] = {'alpha_deg': [], 'Fx_N': [], 'Fy_N': [], 'Mx_Nm': [], 'My_Nm': [], 'Mz_Nm': [], 'mu_eff': []}
    for alpha in np.deg2rad(alpha_deg):
        result = tire.evaluate_static(profile, fz_n=fz, alpha_rad=alpha,
            camber_rad=np.deg2rad(camber_deg), kappa=kappa, vx_mps=speed_mps)
        c = curves[pressure]
        c['alpha_deg'].append(np.rad2deg(alpha)); c['Fx_N'].append(result.fx_n); c['Fy_N'].append(result.fy_n)
        c['Mx_Nm'].append(result.mx_nm); c['My_Nm'].append(result.my_nm); c['Mz_Nm'].append(result.mz_nm)
        c['mu_eff'].append(result.mu_effective)
print('Curvas calculadas:', {p: len(c['alpha_deg']) for p, c in curves.items()})


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 12), constrained_layout=True)
for pressure, group in curves.items():
    axes[0, 0].plot(group['alpha_deg'], group['Fx_N'], label=f'{pressure:g} psi')
    axes[0, 1].plot(group['alpha_deg'], group['Fy_N'], label=f'{pressure:g} psi')
    axes[1, 0].plot(group['alpha_deg'], group['Mx_Nm'], label=f'{pressure:g} psi')
    axes[1, 1].plot(group['alpha_deg'], group['My_Nm'], label=f'{pressure:g} psi')
    axes[2, 0].plot(group['alpha_deg'], group['Mz_Nm'], label=f'{pressure:g} psi')
    axes[2, 1].plot(group['alpha_deg'], group['mu_eff'], label=f'{pressure:g} psi')
for ax, title, ylabel in zip(axes.flat, ['Força longitudinal Fx', 'Força lateral Fy', 'Momento de tombamento Mx', 'Momento de resistência ao rolamento My', 'Momento autoalinhante Mz', 'Atrito resultante efetivo'], ['N', 'N', 'N·m', 'N·m', 'N·m', '—']):
    ax.set(title=title, xlabel='Slip angle α (graus)', ylabel=ylabel)
    ax.grid(True, alpha=.3); ax.legend()
plt.show()


## Sensibilidade à cambagem

Compare Fy e Mx em um ponto de slip angle fixo. Isto é uma varredura do modelo do pneu isolado; não determina por si só a cambagem ótima na suspensão, pois ainda faltam cargas por roda, cinemática, compliance e objetivo veicular.

In [ ]:
pressure_psi = family.pressures_psi[len(family.pressures_psi)//2]
profile = family.at_pressure(pressure_psi)
fz = load_fraction * profile.parameters['LFZO'] * profile.parameters['FNOMIN']
alpha_fixed_deg = float(np.clip(4.0, alpha_min_deg, alpha_max_deg))
camber_min_deg = np.rad2deg(profile.parameters['CAMMIN'])
camber_max_deg = np.rad2deg(profile.parameters['CAMMAX'])
camber_deg_sweep = np.linspace(camber_min_deg, camber_max_deg, 81)
camber = {'camber_deg': [], 'Fy_N': [], 'Mx_Nm': [], 'Mz_Nm': []}
for camber_deg_i in camber_deg_sweep:
    result = tire.evaluate_static(profile, fz_n=fz, alpha_rad=np.deg2rad(alpha_fixed_deg),
        camber_rad=np.deg2rad(camber_deg_i), kappa=kappa, vx_mps=speed_mps)
    camber['camber_deg'].append(camber_deg_i); camber['Fy_N'].append(result.fy_n)
    camber['Mx_Nm'].append(result.mx_nm); camber['Mz_Nm'].append(result.mz_nm)
fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True, constrained_layout=True)
axes[0].plot(camber['camber_deg'], camber['Fy_N'], label='Fy (N)')
axes[1].plot(camber['camber_deg'], camber['Mx_Nm'], color='tab:orange', label='Mx (N·m)')
axes[1].plot(camber['camber_deg'], camber['Mz_Nm'], color='tab:green', label='Mz (N·m)')
axes[0].set_ylabel('Fy (N)'); axes[1].set_ylabel('Momentos (N·m)')
axes[1].set_xlabel('Cambagem γ (graus)')
axes[0].set_title(f'{pressure_psi:g} psi · α={alpha_fixed_deg:g}° · Fz={fz:.0f} N')
for ax in axes: ax.grid(True, alpha=.3); ax.legend()
plt.show()


## Slip ratio e velocidade longitudinal

A varredura abaixo mostra a combinação Fx/Fy/Mz com slip ratio. My é mostrado em função de Vx. No arquivo TTC recebido, os coeficientes QSY1..QSY8 são nulos, então My deve aparecer como zero da parametrização, não como evidência de resistência real nula.

In [ ]:
profile = family.at_pressure(pressure_psi)
p = profile.parameters
fz = load_fraction * p['LFZO'] * p['FNOMIN']
alpha_kappa_deg = alpha_fixed_deg
kappa_min = p['KPUMIN']; kappa_max = p['KPUMAX']
kappa_sweep = np.linspace(kappa_min, kappa_max, 101)
fx_k, fy_k, mz_k = [], [], []
for kappa_i in kappa_sweep:
    result = tire.evaluate_static(profile, fz_n=fz, alpha_rad=np.deg2rad(alpha_kappa_deg),
        camber_rad=np.deg2rad(camber_deg), kappa=kappa_i, vx_mps=speed_mps)
    fx_k.append(result.fx_n); fy_k.append(result.fy_n); mz_k.append(result.mz_nm)
v_sweep = np.linspace(0.0, p['LONGVL'], 81)
my_v = [tire.evaluate_static(profile, fz_n=fz, alpha_rad=np.deg2rad(alpha_kappa_deg),
    camber_rad=np.deg2rad(camber_deg), kappa=0.0, vx_mps=v).my_nm for v in v_sweep]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(kappa_sweep, fx_k, label='Fx (N)'); axes[0].plot(kappa_sweep, fy_k, label='Fy (N)')
axes[0].plot(kappa_sweep, mz_k, label='Mz (N·m)'); axes[0].set(xlabel='Slip ratio κ', title=f'{pressure_psi:g} psi · α={alpha_kappa_deg:g}° · Fz={fz:.0f} N')
axes[0].grid(True, alpha=.3); axes[0].legend()
axes[1].plot(v_sweep, my_v, label='My (N·m)'); axes[1].set(xlabel='Vx (m/s)', title='Momento de resistência ao rolamento')
axes[1].grid(True, alpha=.3); axes[1].legend()
plt.show()


## Revisão local de parâmetros

`revise_profile` cria uma cópia em memória. `save_revision` grava um overlay JSON com hash do `.tir` de origem; o `.tir` original não é alterado e o caminho de saída não pode sobrescrever uma revisão existente. Registre a razão e mantenha o status de validação como não validado até comparar com medições.

O diretório `references/` é local/ignorado pelo Git; salve os overlays ali enquanto o fluxo de revisão permanecer local.

In [ ]:
# Exemplo deliberadamente sem alteração: preencha após documentar a evidência da revisão.
# revised = tire.revise_profile(profile, {'PDY1': novo_valor},
#     reason='Ajuste baseado em comparação X; dados e método registrados em Y',
#     revision_id='R25B-fit-001')
# tire.save_revision(revised, ROOT / 'references/ttc/revisions/R25B-fit-001.json')
# revised = tire.load_revision(profile, ROOT / 'references/ttc/revisions/R25B-fit-001.json')
